# pytRIBS

## The Need For pytRIBS
pytRIBS was designed to aid users of the TIN-based Real-time Integrated Basin Simulator ([tRIBS](https://tribshms.readthedocs.io/en/latest/)) distributed hydrologic model in model setup, execution, and result analysis. Prior to pytRIBS, setting up a tRIBS model could take multiple days, involved a number of proprietary software programs, and was prone to user error. pytRIBS addresses all of these challenges, letting users approach hydrologic modeling in a programmatic and efficient manner.

![tRIBS Workflow](../assets/tRIBS_workflow_horz.png)

**Figure 1.** A tRIBS workflow consists of three main steps: (1) Collate and generate model inputs. (2) Run the model simulation(s). (3) Review and analyze results. The first and last step (denoted by red text and dashed lines) are user intensive activities that challenge reproducibility and are often very time consuming. The asterisks denote steps required for the parallel operation of the model. 

## pytRIBS Design

!['pytRIBS Design'](../assets/pytRIBS_design.png)

**Figure 2.** pytRIBS uses an object-oriented approach to capture the major components and steps required for setting up, simulating, and analyzing a tRIBS model. The preprocessing classes are intended to expedite the first step in setting up a tRIBS model, whereas the simulation classes provide users with tools to effectively run and analyze the model through a Python interface. The Project class is not directly linked to another class as it is limited to storing directory information and meta data. Only select attributes or methods are shown for each class, for a full list of attributes and methods see the associated documentation. 
*Water balance can be calculated for both the basin averaged condition or at individual nodes. 
**Mesh class relies on Preprocess and MeshGenerator classes accessed via these instances.

# Sandbox Start
pytRIBS has a set of workflows and general functions that can be used help creating a tRIBS model. The workflows are generally focused around downloading and processing commonly used datasets like the NLDAS-2 workflow for meteorological data. The general functions while helpful in any scenario are very helpful if datasets not built into pytRIBS are being used. A general model development workflow with pytRIBS is as follows:
1. Setup projection information and generate directory structure.
2. Watershed delineation.
3. Computational mesh development.
4. Soil data processing.
5. Meteorological data processing.
6. Land use data processing.
7. tRIBS input file generation.

The steps above are all contained within this notebook, end-to-end. There are different ways to run a model after it has been setup with pytRIBS. For this sandbox environment the tRIBS code has already been downloaded and compiled, so the model can be run directly from the `Run_Model.ipynb` notebook in this same directory.

To get started with the sandbox we have to first import the Python packages we will need like any other Python code.

## Imports

In [ ]:
# note you can install pytRIBS via pip; see: https://pypi.org/project/pytRIBS/
from pytRIBS.classes import *

In [ ]:
# if you have installed pytRIBS, the following libraries should already be in your environment
import os, sys, shutil
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from shapely.ops import unary_union
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors

## Project Class: Setup Model Information

In [ ]:
name ='SMF'
epsg = 26912
proj = Project(os.getcwd(),name,epsg) # Create an instance of a Project Class

The output below is the folder directory structure that pytRIBS generates where the tRIBS model data is stored.

In [ ]:
proj.directories

In [ ]:
proj.meta

The code below code copies in the needed pre-processed data sets and sets file paths to required data.

In [ ]:
# Set the file path to our DEM
dem = '../init_data/USGS_1m_clip.tif'

# Copy over our pre-generated Soil and Land Use ID rasters into our project directories
landuse_ras = '../init_data/LandUse.asc'
shutil.copy(landuse_ras, proj.directories['land'])
landuse_ras = f"{proj.directories['land']}/{os.path.basename(landuse_ras)}"

soil_ras = '../init_data/ADOT_SoilTypes.asc'
shutil.copy(soil_ras, proj.directories['soil'])
soil_ras = f"{proj.directories['soil']}/{os.path.basename(soil_ras)}"

## Mesh Class: Process DEM and Generate Mesh

### Preprocessing
Here initiate the class with arguments for the Preprocessing and MeshGeneration component classes. 

Before we make a mesh we need to delineate the watershed and to do that we use the code below to define some of the information required.

In [ ]:
# Preprocessing Arguments
verbose_mode=False# suppresses whitebox output

# UTM coordinates for outlet/pour point
x = 394456
y = 3686772

snap_dis = 100 # allowable distance in m for snapping outlet points to stream network

# This area threshold for determining streams for watershed delineation
# Lets initially try 0.02 sqkm (2e4 m^2), this value is important because it determines the number of streams in tRIBS mesh
threshold_area = 2e4 # area in m^2 for determining stream network

# tuple of preprocessing arguments to be passed into the the mesh class
preprocess_args = ([x,y],snap_dis, threshold_area, dem, verbose_mode,proj.meta,proj.directories['preprocessing'])

In [ ]:
# Mesh Generation Arguments
# output directory can be specified in Preproceesing, but if not provided preprocessing is the default directory 
output_dir = proj.directories['preprocessing'] 

path_to_raster = f'{output_dir}/{name}_clipped_ext.tif' # these are default outputs but can be further modified. See documentation.
path_to_watershed = f'{output_dir}/{name}_boundary.shp'
path_to_stream_network = f'{output_dir}/{name}_stream.shp'
path_to_outlet = f'{output_dir}/{name}_outlet.shp'
maxlevel= None # This can be set to none, if so the maximum level possible will be used. 

mesh_generation_args = (path_to_raster, path_to_watershed, path_to_stream_network, path_to_outlet, maxlevel)

The code below executes the watershed delineation and saves all of the results to the filepaths listed above.

In [ ]:
tmesh = Mesh(preprocess_args=preprocess_args, generate_mesh_args=mesh_generation_args,meta=proj.meta,mesh_dir=proj.directories['mesh'])

Now, we have created all the files necessary for generating a mesh. Below we provide a visualization of these data. Note you could read this data in independently then plot it, but they are all ready assigned to ```tmesh.mesh_generator``` or alternatively can be used from the MeshGeneration class.

#### Example Figure 1: Preprocessing products produced by pytRIBS

In [ ]:
dem_data = tmesh.mesh_generator.data
valid_data = dem_data[dem_data > -500] 
vmin, vmax = np.percentile(valid_data, [2, 98])
print(f"Plotting with elevation limits: {vmin:.1f}m to {vmax:.1f}m")

# Create the Plot
fig, ax = plt.subplots(figsize=(10, 8))

extent = tmesh.mesh_generator.get_extent()

# Display the image with manual limits (vmin/vmax)
img = ax.imshow(
    dem_data,
    extent=extent,
    cmap='terrain',
    vmin=vmin,
    vmax=vmax 
)

# Add colorbar
cbar = fig.colorbar(img, ax=ax, orientation='vertical', shrink=0.8)
cbar.set_label('Elevation (m)', fontsize=14)
cbar.ax.tick_params(labelsize=12)

# Plot watershed boundary (using Geopandas .plot)
tmesh.mesh_generator.watershed.plot(
    ax=ax,
    facecolor='none',
    edgecolor='red',
    linewidth=2,
    zorder=2 # draws on top of raster
)

# Plot stream network
tmesh.mesh_generator.stream_network.plot(
    ax=ax,
    linewidth=1.5,
    color='blue', # Rivers are usually blue!
    zorder=2
)

# Plot outlet
tmesh.mesh_generator.outlet.plot(
    ax=ax,
    color='yellow',
    marker='*',
    markersize=200, # Made it a bit bigger to see
    edgecolor='black',
    zorder=3
)

# Set axis labels and title
ax.set_xlabel('Easting (UTM)', fontsize=14)
ax.set_ylabel('Northing (UTM)', fontsize=14)
ax.set_title('Watershed, Stream Network, and Outlet', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)

# Create manual legend
legend_elements = [
    Line2D([0], [0], color='red', lw=2, label='Watershed Boundary'),
    Line2D([0], [0], color='blue', lw=1.5, label='Stream Network'),
    Line2D([0], [0], color='yellow', marker='*', markeredgecolor='black', 
           markersize=15, linestyle='None', label='Outlet')
]

ax.legend(handles=legend_elements, loc='upper left', fontsize=12, framealpha=0.9)

# Add a buffer to zoom out slightly 
# Get the bounding box coordinates of the watershed vector
minx, miny, maxx, maxy = tmesh.mesh_generator.watershed.total_bounds

# Calculate the width and height of the watershed
width = maxx - minx
height = maxy - miny

# Define a buffer percentage (e.g., 10% on each side)
buffer_pct = 0.01

# Apply the new limits with the buffer
ax.set_xlim(minx - (width * buffer_pct), maxx + (width * buffer_pct))
ax.set_ylim(miny - (height * buffer_pct), maxy + (height * buffer_pct))

plt.tight_layout()
plt.show()

### Mesh Generation
With the data from previous steps we can generate a locally refined mesh rapidly and easily using multiple methods combined into the same workflow. 

1. Harr wavelet transform: this piece of the workflow is a method applied to the DEM to select significant points. Significance meaning how important is that location in describing the terrain.
2. Planar Straight Line Graph (PSLG). Using a open-source python tools we can generate a TIN mesh directly using our watershed boundary and streamlines as breaklines, embedding them into the mesh. Here we can also specify a number of constraints on the mesh generation like: what is the minimum interior angle allowed in the mesh?

Below is an example for executing the workflow. The parameters set produce a usable mesh but they can also be modified (to improve detail,reduce computational time, or experimentation). Another component of the mesh generation lies in the delineation step above with the stream network area threshold. This can be changed to add or reduce the number of streams which will embedded put into the mesh.

In [ ]:
# This parameter is a important piece of the mesh generation in pytRIBS.
# It controls how many points will be classified as significant thus kept versus in-significant points that don't need to be included in the mesh.
# Lower values means more points kept (denser mesh). Range from 0 to 1 (lower = denser).
threshold = 0.2

Mesh parameters

`threshold` controls  initial interior-point density. After those points are created they are fed into triangulation algorithm to make the TIN mesh. The variables below are some of the constraint we can supply to the triangulation algorithm.

In [ ]:
boundary_buffer_dist = 40.0   # meters: watershed buffered outward to the no-flow boundary
boundary_spacing     = 70.0   # meters: node spacing on the boundary ring
stream_point_spacing = 40.0   # meters: node spacing along streams
stream_clear_radius  = 2.0   # meters: interior points this close to a stream are removed
quality_opts         = 'q15a15000'   # Triangle quality: q = min interior angle of triangle (degrees), a = max area (square meters)

Build the mesh

Writes the mesh created above into a set of 4 files: `{name}_mesh.nodes/.edges/.tri/.z` into `proj.directories['mesh']`. tRIBS reads these 4 files at the start of the simulation. The `diagnostics=True` option below writes inspection shapefiles (triangles, points, etc) for QA into the preprocessing folder. 

After the code block below is ran a large printout below will be generated giving more details about the mesh created.

In [ ]:
tmesh.build_mesh(
    method='pslg',
    threshold=threshold,
    boundary_buffer_dist=boundary_buffer_dist,
    boundary_spacing=boundary_spacing,
    stream_point_spacing=stream_point_spacing,
    stream_clearance_radius=stream_clear_radius,
    mesh_quality_opts=quality_opts,
    diagnostics=True,                             # Set to true this write shapfiles of the TIN mesh
    separate_parallel_streams=True,               # With true a second pass is ran, essentially the TIN mesh is checked to remove potential issue then created again
)

## Soil Class: Obtain and Generate tRIBS Soil Parameters

There are two methods to provide tRIBS soil parameters: a soil ID map and table or gridded parameter rasters. For SMF since we have the ADOT soil data as polygons we will use the simpler method, a soil ID map with a lookup table. Note pytRIBS has functionality to download gridded soil datasets that are publicly available but not used in this exercise.

In [ ]:
soil = Soil(meta=proj.meta)

Another component of the soil class is the initial ground water table and depth to bedrock.

Bedrock data can be quite difficult to find but it is becoming more available from Digital Soil Mapping (DSM) products. For this model we will apply the [SOLUS](https://www.nrcs.usda.gov/resources/data-and-reports/soil-landscapes-of-the-united-states-solus) dataset which has already been pre-processed for use here. Due to the shallow soils here and the dry conditions we are assuming that the initial groundwater table is essentially at the bedrock. To do that we multiplied our depth to bedrock raster by 95% i.e. if the depth to bedrock is 1m then the initial groundwater depth is 0.05m above the bedrock.

All we need to do is copy over the data into our model directory.

In [ ]:
# Depth to Bedrock
shutil.copy('../init_data/SOLUS_Bedrock_m.asc', proj.directories['soil'])
soil.bedrockfile['value'] = f"{proj.directories['soil']}/SOLUS_Bedrock_m.asc"

# Initial Groundwater Table
shutil.copy('../init_data/InitGW_95pct_mm.asc', proj.directories['soil'])
soil.gwaterfile['value'] = f"{proj.directories['soil']}/InitGW_95pct_mm.asc"

Now that the soil class is created we can tell pytRIBS that we already have the soil data. Instead of using the tools built-in to pytRIBs to download and prepare soil data.

In [ ]:
# Point pytRIBS at the soil ID map and where the new soil table will be written
soil.soilmapname['value'] = soil_ras
soil.soiltablename['value'] = f"{proj.directories['soil']}/soils.sdt"

In [ ]:
# Texture labels per class ID (from the ADOT Soil dataset)
# This looks complicated but really we are just assigning soil ID number a specific name from the ADOT dataset
soil_textures = {
      '1': 'RS', '2': 'CO', '3': 'CeD', '4': 'EbD', '5': 'Cb',
}
soil_map = InOut.read_ascii(soil.soilmapname['value'])

# Build an empty table straight from the soil map's class IDs
soil_table = soil.create_soil_table_from_map(textures=soil_textures)

Now that we have a blank soil table we can fill it up with the parameter values from the ADOT dataset.

In [ ]:
# Lets define the soil parameters. Some are parameters are specific to each soil class, some will be static across all classes
# More information here: https://tribshms.readthedocs.io/en/latest/man/Model_Parameters_Forcings.html

# Define Hydraulic Parameters for Specific Soil Classes
# Here we map the specific soil IDs found in the ADOT dataset (MUSYM) to physical hydraulic parameters.
# In a research setting, these might be derived from pedotransfer functions (like Rosetta) or field tests.
# For this exercise, we are using the values from the ADOT data as a starting point.

# Parameters:
# - Ks: Saturated hydraulic conductivity (mm/hr)
    # This is an important calibration parameter
# - thetaS: Saturation soil moisture content (porosity) (-)
# - thetaR: Residual soil moisture content (-)
# - m: Pore size distribution index (Lambda) (-)
# - PsiB: Air entry/Bubbling pressure (mm)
# - f: Conductivity decay parameter (1/mm) - Controls how fast Ksat decreases with depth
    # This is an important calibration parameter
# - n: Porosity (-)

# Define a dictionary to map your ADOT Soil IDs to parameters.
soil_param_lookup = {
    # Soil: RS
    '1': {
        'Ks': 3.6,      # From ADOT dataset, KEY calibration parameter
        'thetaS': 0.40,  # From ADOT dataset
        'thetaR': 0.06,  # Initial guess we are using 0.5*wilting point
        'm': 0.38,       # Value selected from Rawls et al 1984.
        'PsiB': -390,    # From ADOT dataset
        'f': 0.001,       # Assume initial value, KEY calibration parameter
        'n': 0.40        # Initial guess we are using this is equal to thetaS
    },
    # Soil: CO
    '2': {'Ks': 2.8,  'thetaS': 0.40, 'thetaR': 0.05, 'm': 0.25, 'PsiB': -401, 'f': 0.001, 'n': 0.40},
    # Soil: CeD
    '3': {'Ks': 6.6,  'thetaS': 0.40, 'thetaR': 0.05, 'm': 0.25, 'PsiB': -183, 'f': 0.001, 'n': 0.40},
    # Soil: EbD
    '4': {'Ks': 1.0,  'thetaS': 0.42, 'thetaR': 0.10, 'm': 0.20, 'PsiB': -450, 'f': 0.001, 'n': 0.42},
    # Soil: Cb
    '5': {'Ks': 17.3, 'thetaS': 0.39, 'thetaR': 0.03, 'm': 0.18, 'PsiB': -117, 'f': 0.001, 'n': 0.39},
}

# Now that we defined our parameter values above lets populate the soil data table with thos values
for cls in soil_table:
    # Constant parameter values for all soil classes
    cls.update({'As': 1, 'Au': 1, 'ks': 0.7, 'Cs': 1.4e6})
    cls.update(soil_param_lookup[cls['ID']])
    print(f"Assigned parameters for Soil ID {cls['ID']} ({cls['Texture']})")

# Write the populated soil table
soil.write_soil_table(soil_table, soil.soiltablename['value'], textures=True)


Now that we have the soil table lets tell pytRIBS that in the main input file we will be using soil option 0 that corresponds to using the soil table

In [ ]:
soil.optsoiltype['value'] = 0

#### Example Figure 3: Example soil classification map generated from pytRIBS Soil Class

In [ ]:
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import numpy as np

data = soil_map['data']
transform = soil_map['profile']['transform']

# Calculate Extent (Standard GeoTransform logic)
x_min = transform[2]
x_max = x_min + soil_map['profile']['width'] * transform[0]
y_max = transform[5]
y_min = y_max + soil_map['profile']['height'] * transform[4] 
extent = [x_min, x_max, y_min, y_max]

# Mask  Data
masked_data = np.ma.masked_less(data, 0)

# Find unique soil classes for the legend
unique_classes = np.unique(masked_data.compressed())
num_classes = len(unique_classes)

print(f"Found {num_classes} unique soil classes: {unique_classes}")

# Create Discrete Colormap
cmap = plt.get_cmap('cividis', num_classes)
norm = colors.BoundaryNorm(np.arange(len(unique_classes) + 1), cmap.N) 

# Plotting
fig, ax = plt.subplots(figsize=(12, 10))
img = ax.imshow(
    masked_data,
    extent=extent,
    cmap=cmap,
    interpolation='nearest' # Prevents blurring between classes
)

# 6. Watershed Boundary
tmesh.mesh_generator.watershed.plot(
    ax=ax, 
    facecolor='none', 
    edgecolor='red', 
    linewidth=2,
    label='Watershed Boundary'
)

# 7. Custom Colorbar for Categorical Data
# We tell the colorbar to only show ticks at the center of each discrete color
cbar = plt.colorbar(img, ax=ax, ticks=unique_classes)
cbar.set_label('Soil Class ID', fontsize=14)
cbar.ax.tick_params(labelsize=12)

# Set the limits of the image to the watershed bounds
bounds = tmesh.mesh_generator.watershed.total_bounds
ax.set_xlim([bounds[0], bounds[2]])
ax.set_ylim([bounds[1], bounds[3]])

# Labels
ax.set_xlabel('Easting (UTM)', fontsize=14)
ax.set_ylabel('Northing (UTM)', fontsize=14)
ax.set_title(f'Soil Classification Map', fontsize=16)

plt.tight_layout()
plt.show()


## Met Class: Assign Meteorological Forcing Data
Meteorological forcing for the tRIBS model can be supplied either as point-based station data or as raster data. Managing these data manually can be time-consuming and susceptible to errors. To simplify this process, the Met class allows users to specify the filepaths to already generated forcings files or efficiently obtain meteorological forcing data from the North American Land Data Assimilation System (NLDAS) for a specified location and time period. 

For this notebook forcing data has already been prepared so we will not be using the NLDAS workflow. First we need to tell pytRIBS the files paths where our pre-processed forcing will be located:

In [ ]:
met = Met(meta=proj.meta) # Initializing the pytRIBS Met Class
met.hydrometstations['value'] = "../init_data/met/Master_Met.sdf" # File path to where station data file will live
met.gaugestations['value'] = "../init_data/met/Master_Precip.sdf" # File path to where station data file will live

Meteorological forcings for SMF have already been generated using the `Generate_Met_Forcing` notebook. If you want to change the simulation's length or time period, rerun that notebook to regenerate the forcing files.

More details on the required tRIBS forcing files are located here: [Model Forcings](https://tribshms.readthedocs.io/en/latest/man/Model_Forcings.html#model-forcings)

## Land Class: Create Land Cover Map and Assign tRIBS Land Cover Parameters
tRIBS supports spatially varying land cover parameters, which can be provided through a classification map and table, raster data, or a combination of both. These inputs may remain static over time or vary across different periods (e.g., monthly, seasonally, annually). The Land class offers various attributes and methods to manage these parameters, although the availability and resolution of land cover data often differ significantly between applications, necessitating more user input. Consequently, the Land class does not offer an all-encompassing workflow but instead provides a set of tools to manage data from various sources and helper functions to generate the necessary input files for a tRIBS simulation. 

For this notebook we have already pre-processed the National Land Cover Database dataset (NLCD) from 2014 for the watershed, which we already copied into the land folder at the start of this notebook. The map is broken down into `3` classes:  
`1 - South Facing Slopes`  
`2 - North Facing Slopes`  
`3 - Roadway`

While the NLCD only has the `2` land cover classes (shrub/scrub and the roadway) the shrub/scrub class was further separated into 2 classes because of what was observed during field visits to the site.

We will set initial vegetation parameters based on values from prior tRIBS projects. More details on the required tRIBS parameters is located here: [Model Parameters](https://tribshms.readthedocs.io/en/latest/man/Model_Parameters.html#model-parameters)

In [ ]:
# Initialize the pytRIBS Land class
land = Land(meta=proj.meta)

In [ ]:
# Tell pytRIBS where the landcover map that we already copied over exists
land.landmapname['value'] = f"{proj.directories['land']}/LandUse.asc"
land.landtablename['value'] = f"{proj.directories['land']}/land_use_params.ldt"

In [ ]:
# Similar to how we set up the Soil Table we will make a dictionary of parameter values then use the pytRIBS writing function

# Parameters (Table 3.2, tRIBS docs -- see link above):
# - P: Free throughfall coefficient, Rutter method (-)
# - S: Canopy field capacity, Rutter method (mm)
# - K: Drainage coefficient, Rutter method (mm/hr)
# - b2: Drainage exponent, Rutter method (1/mm)
# - Al: Albedo (-)
# - h: Vegetation height (m)
# - Kt: Optical transmission coefficient (-)
# - Rs: Stomatal resistance (s/m)
# - V: Vegetation fraction (-)
# - LAI: Leaf Area Index (-)
# - theta*_s: Stress threshold for evaporation, [theta_R to theta_S] (-)
# - theta*_t: Stress threshold for transpiration, [theta_R to theta_S] (-)

# Define the Land Use Parameter Dictionary
land_param_lookup = {
    # Class 1: South Facing Slopes (e.g., Scrub/Shrub)
    '1': {
        'P': 0.4,       # Sparse canopy, lots of throughfall
        'S': 1.5,       # Low storage
        'K': 0.12,      # Drainage Coeff
        'b2': 4.7,      # Drainage Exp
        'Al': 0.18,     # Lower albedo (darker)
        'h': 1,       # Short vegetation
        'Kt': 0.4,      # Optical transmission
        'Rs': 120,      # High resistance (desert plants)
        'V': 0.15,       # Vegetation Fraction 
        'LAI': 1.5,     # Low Leaf Area Index
        'theta*_s': 0.37, # Soil Mositure Stress threshold
        'theta*_t': 0.30
    },
    # Class 2: North Facing Slopes 
    '2': {
        'P': 0.4,       # Sparse canopy, lots of throughfall
        'S': 1.5,       # Low storage
        'K': 0.12,      # Drainage Coeff
        'b2': 4.7,      # Drainage Exp
        'Al': 0.18,     # Lower albedo (darker)
        'h': 1,       # Short vegetation
        'Kt': 0.4,      # Optical transmission
        'Rs': 120,      # High resistance (desert plants)
        'V': 0.30,       # Vegetation Fraction (60% bare soil)
        'LAI': 1.5,     # Low Leaf Area Index
        'theta*_s': 0.37, # Stress threshold
        'theta*_t': 0.30
    },
    # Class 3: Roadway / Developed (Impervious)
    '3': {
        'P': 0.99,       # No canopy, rain hits ground instantly
        'S': 0.01,       # No storage
        'K': 0.001, 
        'b2': 0.001, 
        'Al': 0.15,     # Asphalt/Concrete albedo
        'h': 0.01,      # Near zero height
        'Kt': 0.99,     # All light hits ground
        'Rs': 9999,     # Infinite resistance (no transpiration)
        'V': 0.01,      # No vegetation
        'LAI': 0.01, 
        'theta*_s': 0.37, 
        'theta*_t': 0.30
    },
}

# Construct the list of dictionaries for the write function
landuse_list = []

for lu_id, params in land_param_lookup.items():
    # Create a copy so we don't mess up the original dict
    row = params.copy()
    
    # Add the ID to the row (required by the write function)
    row['ID'] = lu_id
    
    # Add dummy interception Parameters (a and b1) for interception scheme we are not using.
    row['a'] = -9999
    row['b1'] = -9999
    
    landuse_list.append(row)

# Write the Table
land.write_landuse_table(landuse_list, land.landtablename['value'])

print(f"Land Use Table written to: {land.landtablename['value']}")

## Model Class: Pre-flight Check and Model Simulation
The Model Class allows users to modify tRIBS model inputs, validate that all inputs are appropriate for the chosen options, and run the tRIBS model, either directly, or via the Docker SDK for Python if the model isn't installed locally. It can be initialized with or without preprocessing classes, providing a flexible approach to numerical experiments, and also supports manual setup or starting from an existing input file.

For this notebook we will be using the Model Class to build the input file. Since this sandbox environment already has tRIBS compiled and on the system path, the model can be run directly without Docker — that happens in the `Run_Model.ipynb` notebook in this same directory.

In [ ]:
model = Model(met=met,land=land,soil=soil,mesh=tmesh,meta=proj.meta)

The last set of parameters we need to define is for setting up the channel transmission losses. There are a couple ways to set this up in tRIBS. For this exercise we will use the Constant Loss method.

This method requires 1 input parameter:

    - CHANNELCONDUCTIVITY is the steady-state bed conductivity [mm/hr]

This parameter value is the same for all of the reaches across the entire model.

Or you can test the model with no transmission losses at all by turning off the option.

In [ ]:
# First we need to tell pytRIBS whether the transmission losses are off (0) or on (1)
model.optpercolation['value'] = 1 # Lets turn it on and see how the model performs
model.channelconductivity['value'] = 70 # tRIBS the saturated conductivity of the channel material in mm/hr

Lets specify some additional parameter values for the tRIBS kinematic routing scheme

  - KINEMVELCOEF is a direct multiplier on hillslope velocity, doubling it halves hillslope travel time
  - FLOWEXP controls how much velocity scales with discharge, lower values mean more constant velocity regardless of storm intensity
  - CHANNELROUGHNESS is the Manning's n roughness coefficient for the channel
  - CHANNELWIDTHCOEFF controls the width of the channel network

We'll set KINEMVELCOEF and FLOWEXP to some standard starting values for tRIBS. Then set the roughness to value that is more aligned with what we saw in the field but it can still be adjusted.

In [ ]:
model.kinemvelcoef['value'] = 3
# kinemvelcoef = 3 gives ~0.03 m/s (slow), kinemvelcoef = 10 gives ~0.09 m/s, kinemvelcoef = 50 gives ~0.45 m/s.

model.flowexp['value'] = 0.3
# General range is from 0.3-0.6
model.channelroughness['value'] = 0.04
model.channelwidthcoeff['value'] = 2.33
# The smaller this value is the faster the flow will move through the channel. General range is 0.5-2.5

In [ ]:
# mesh
model.parallelmode['value'] = 0 # Running the model in serial mode, not parallel
model.optmeshinput['value'] = 1 # We are using mesh input option 1 for a pre-generated mesh

# soil 
model.optbedrock['value'] = 1 # we are using our gridded bedrock depth map

# Snow Module
model.optsnow['value'] = 0 # turn off the snow module because we do not need it where we're going

#land
model.optlanduse['value'] = 0 # Only parameters are from the land use table so option 0

#simulation variables
model.startdate['value'] = '08/01/2014/00/00'
model.runtime['value'] = 480 # simulation length in hours
model.outfilename['value'] = f"{proj.directories['results']}/{name}"

# Forcings
# pytRIBS defaults to 1-hour which is the most often common value but our precipitaiton is in 15-min increments
model.rainintrvl['value'] = 0.25 # Rain input time interval in hours

Below we need to specify which voronoi polygons we want to get additional detailed information on.

In [ ]:
# Create node list file for outputing Pixel file output (detailed timeseries for individual polygons)
node_ids = [1960, 1547] # these are example node IDs with no real meaning but can be changed to any ID value
model.write_node_file(node_ids,'data/model/pnodes.dat')
model.nodeoutputlist['value'] = 'data/model/pnodes.dat'

# Create node list file for outputing internal streamflow results (*.qout)
node_ids = [3202] # these are example node IDs with no real meaning but can be changed to any ID value
model.write_node_file(node_ids,'data/model/qnodes.dat')
model.outletnodelist['value'] = 'data/model/qnodes.dat'

With everything configured, `check_paths()` runs pytRIBS's pre-flight validation, it checks that every file path the input file references (mesh, soil, land, met, node lists) actually exists and that the chosen options are internally consistent, before we write anything out.

In [ ]:
model.check_paths()

In [ ]:
input_file = f'{name}.in'

In [ ]:
model.write_input_file(input_file)

The model is now fully configured and `SMF.in` has been written. Head to `Run_Model.ipynb` in this same directory to run the simulation and look at the results.